In [10]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import torch                       # <--- MỚI: Thêm PyTorch
import torch.nn as nn              # <--- MỚI
import torch.optim as optim        # <--- MỚI
import torchvision                 # <--- MỚI
import torchvision.transforms as transforms # <--- MỚI
from scipy.signal import butter, filtfilt, hilbert
import os
import pickle
import gc

CFG = {
    "fs": 1.0,            # <--- SỬA: Coi mỗi ảnh là 1 bước thời gian (sample rate = 1)
    "batch_size": 256,    # <--- MỚI: Đóng vai trò là "Time duration" cho thuật toán Psi
    "num_classes": 10,    # <--- MỚI: CIFAR-10 có 10 lớp
}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "legend.frameon": False,
    }
)

def format_label(key: str) -> str:
    """
    Convert internal identifiers to formatted labels.
    """
    special_map = {
        "pac": "PAC"
    }

    words = key.replace("_", " ").split()
    pretty_words = []
    for w in words:
        lw = w.lower()
        if lw in special_map:
            pretty_words.append(special_map[lw])
        elif w.isupper():
            pretty_words.append(w)
        else:
            pretty_words.append(w.capitalize())
    return " ".join(pretty_words)

In [2]:
def setup_densenet_hooks(model):
    """
    Gắn hook vào 4 Dense Blocks của DenseNet-121.
    DenseNet-121 gồm:
    - Dense Block 1 (6 layers)  -> Tương ứng Layer 1 (Early)
    - Dense Block 2 (12 layers) -> Tương ứng Layer 2 (Mid-Early)
    - Dense Block 3 (24 layers) -> Tương ứng Layer 3 (Mid-Late)
    - Dense Block 4 (16 layers) -> Tương ứng Layer 4 (Late)
    """
    activations = {}
    
    def get_activation(name):
        def hook(model, input, output):
            activations[name] = output.detach()
        return hook

    # Mapping từ tên chuẩn của code bạn sang tên layer trong DenseNet
    layer_mapping = {
        'layer1': 'denseblock1',
        'layer2': 'denseblock2',
        'layer3': 'denseblock3',
        'layer4': 'denseblock4'
    }

    print(f"Hooking DenseNet layers: {list(layer_mapping.values())}")
    
    # DenseNet trong torchvision lưu các block bên trong `model.features`
    for key, block_name in layer_mapping.items():
        if hasattr(model.features, block_name):
            # Lấy module con (VD: model.features.denseblock1)
            module = getattr(model.features, block_name)
            module.register_forward_hook(get_activation(key))
        else:
            print(f"Warning: Không tìm thấy {block_name} trong model.features")
            
    return activations

def gather_multiscale_activations(activations_dict, samples_per_layer=32):
    """
    Thực hiện ALGORITHM 1 (Lines 7-9):
    1. Lấy activation từ 4 layers.
    2. Global Average Pooling (giảm chiều không gian H,W).
    3. Random sample 32 channels/layer.
    4. Concatenate thành matrix [Batch, 128].
    """
    collected_features = []
    layers = ['layer1', 'layer2', 'layer3', 'layer4']
    
    for layer_name in layers:
        if layer_name not in activations_dict:
            continue
            
        # Raw shape: [Batch, Channels, H, W]
        act = activations_dict[layer_name]
        
        # Global Average Pooling: [B, C, H, W] -> [B, C]
        # Đại diện cho trạng thái kích hoạt trung bình của feature map
        if len(act.shape) > 2:
            act = torch.mean(act, dim=[2, 3]) 
            
        # Sampling: Lấy ngẫu nhiên 32 channels
        n_channels = act.shape[1]
        
        # Dùng generator để đảm bảo tính ngẫu nhiên nhưng đồng nhất trong 1 batch
        if n_channels >= samples_per_layer:
            idx = torch.randperm(n_channels, device=act.device)[:samples_per_layer]
            act = act[:, idx]
        else:
            # Trường hợp hiếm (layer quá nhỏ): giữ nguyên
            pass
            
        collected_features.append(act)
    
    # Concatenate theo chiều Channels (dim 1) -> [Batch, 128]
    if not collected_features:
        return None
        
    final_matrix = torch.cat(collected_features, dim=1)
    return final_matrix

def prepare_cnn_activations(activations, labels, smooth_window=5):
    """
    ADAPTER V3 (FINAL):
    Biến đổi Matrix Activation thành dạng sóng giả lập (Pseudo-time Series).
    Bao gồm: Sắp xếp theo Class -> Local Centering -> Smoothing.
    
    Input: activations [Batch, 128], labels [Batch]
    Output: numpy array [128, Batch] (Channels x Time)
    """
    # 1. Chuyển về CPU numpy
    if isinstance(activations, torch.Tensor):
        data = activations.detach().cpu().numpy()
    else:
        data = activations
        
    if isinstance(labels, torch.Tensor):
        lbls = labels.detach().cpu().numpy()
    else:
        lbls = labels
    
    # 2. Sắp xếp theo nhãn (Sort by Label) -> Tạo cấu trúc thời gian giả lập
    sort_idx = np.argsort(lbls)
    data_sorted = data[sort_idx] # Shape: (Time, Channels)
    
    # 3. Class-wise Centering (Quan trọng cho Metastability)
    # Loại bỏ giá trị trung bình của từng class để làm nổi bật dao động
    df_temp = pd.DataFrame(data_sorted)
    df_temp['label'] = lbls[sort_idx]
    
    # Trừ đi mean của từng nhóm class
    data_detrended = df_temp.groupby('label').transform(lambda x: x - x.mean()).values
    
    # 4. Smoothing (Quan trọng cho Hurst)
    # Khử nhiễu gai góc của ReLU, giúp H tăng lên mức 0.6-0.8
    # Window=5 là đủ để mượt mà vẫn giữ được biến động nhanh
    data_smoothed = pd.DataFrame(data_detrended).rolling(
        window=smooth_window, center=True, min_periods=1
    ).mean().values
    
    # 5. Transpose về (Channels, Time) để phù hợp hàm DFA/Hilbert
    return data_smoothed.T

In [3]:
# ---------------------------------------------------------------------
# CORE METRIC UTILITIES (DFA, LZ, MI)
# ---------------------------------------------------------------------

def dfa_hurst(x, min_win=16, max_win=None, n_win=10):
    """
    DFA-based Hurst exponent.
    SỬA ĐỔI: Thêm check độ dài dữ liệu để tránh crash nếu Batch Size nhỏ.
    """
    x = np.asarray(x)
    N = x.size
    
    # Safety check: Nếu Batch Size < 32, DFA không tính được chính xác -> Trả về 0.5 (Random walk)
    if N < 32: 
        return 0.5
        
    if max_win is None:
        max_win = N // 4
        
    # Safety check: Đảm bảo max_win luôn lớn hơn min_win
    if max_win <= min_win:
        max_win = N // 2

    y = np.cumsum(x - x.mean())
    
    # Tạo danh sách các cửa sổ (log-scale)
    s_vals = np.unique(
        np.logspace(np.log10(min_win), np.log10(max_win), n_win, dtype=int)
    )
    
    F = []
    for s in s_vals:
        if s < 4: continue
        
        n_segments = N // s
        if n_segments < 2: continue
        
        rms = []
        for i in range(n_segments):
            seg = y[i * s : (i + 1) * s]
            t = np.arange(s)
            p = np.polyfit(t, seg, 1)
            trend = np.polyval(p, t)
            detrended = seg - trend
            rms.append(np.sqrt(np.mean(detrended**2)))
            
        if rms:
            F.append(np.mean(rms))
            
    F = np.array(F)
    if len(F) < 2:
        return 0.5

    s_use = s_vals[: len(F)]
    # Fit đường thẳng trên đồ thị log-log
    coeffs = np.polyfit(np.log(s_use), np.log(F), 1)
    H = coeffs[0]
    return float(H)


def lempel_ziv_complexity(binary_seq):
    """
    Lempel–Ziv complexity for a 1D binary sequence (0/1).
    Giữ nguyên.
    """
    s = "".join(str(int(b)) for b in binary_seq)
    i, c, l = 0, 1, 1
    n = len(s)
    while True:
        if i + l > n:
            c += 1
            break
        sub = s[i : i + l]
        if sub in s[:i]:
            l += 1
        else:
            i += l
            c += 1
            l = 1
        if i + l > n:
            break
    return c / n


def mutual_information_phase_amp(phase, amp, n_bins=12):
    """
    Mutual information between phase and amplitude.
    Giữ nguyên (có thể dùng để tính toán phụ, dù Psi chính chỉ cần H và M).
    """
    phase = np.asarray(phase)
    amp = np.asarray(amp)

    phase_bins = np.linspace(-np.pi, np.pi, n_bins + 1)
    amp_bins = np.quantile(amp, np.linspace(0, 1, n_bins + 1))
    phase_d = np.digitize(phase, phase_bins) - 1
    amp_d = np.digitize(amp, amp_bins) - 1

    valid = (
        (phase_d >= 0) & (phase_d < n_bins) & 
        (amp_d >= 0) & (amp_d < n_bins)
    )
    phase_d = phase_d[valid]
    amp_d = amp_d[valid]
    
    if len(phase_d) == 0: return 0.0

    joint, _, _ = np.histogram2d(phase_d, amp_d, bins=(n_bins, n_bins))
    if joint.sum() == 0: return 0.0
    
    joint = joint / joint.sum()
    px = joint.sum(axis=1)
    py = joint.sum(axis=0)

    mi = 0.0
    for i in range(n_bins):
        for j in range(n_bins):
            pxy = joint[i, j]
            if pxy > 0 and px[i] > 0 and py[j] > 0:
                mi += pxy * np.log(pxy / (px[i] * py[j]))
    return float(mi)

In [4]:
# ---------------------------------------------------------------------
# METRIC COMPUTATION (Raw Components only)
# ---------------------------------------------------------------------

def compute_raw_metrics(X, Hopt=0.7, sigma_H=0.15):
    """
    Compute raw Heff and M.
    NOTE: Does NOT compute final Psi because Psi requires population Z-scoring.
    """
    C, T = X.shape

    # --- 1. Hierarchical Integration (H_eff) ---
    H_vals = []
    for ch in range(C):
        try:
            h = dfa_hurst(X[ch])
        except:
            h = 0.5
        H_vals.append(h)
        
    H_raw = float(np.mean(H_vals))
    
    # Gaussian Tuning (Eq. 2 in paper)
    Heff = np.exp(-((H_raw - Hopt) ** 2) / (2.0 * sigma_H**2))

    # --- 2. Metastability (M) ---
    # Centering signal
    X_centered = X - np.mean(X, axis=1, keepdims=True)
    
    # Hilbert Transform to get Phase
    analytic_signal = hilbert(X_centered, axis=-1)
    phases = np.angle(analytic_signal)
    
    # Kuramoto Order Parameter R(t) (Eq. 3 in paper)
    R_t = np.abs(np.mean(np.exp(1j * phases), axis=0))
    
    # Metastability (Eq. 4 in paper)
    M = float(np.std(R_t))

    return {
        "H_raw": H_raw, 
        "Heff": Heff, 
        "M": M
    }

In [5]:
def summarize_experiment_results(experiment_log):
    """
    Tổng hợp kết quả từ quá trình training.
    Input: List các dictionary lưu trữ theo từng epoch.
    Output: DataFrame đã tính toán Psi chuẩn hóa (Z-score).
    """
    df = pd.DataFrame(experiment_log)
    
    # 1. Tính trung bình và độ lệch chuẩn của toàn bộ quá trình (Population stats)
    # Để dùng cho công thức Z-score (Eq. 5 trong bài báo)
    mu_H = df['Heff'].mean()
    std_H = df['Heff'].std() if df['Heff'].std() > 0 else 1e-9
    
    mu_M = df['M'].mean()
    std_M = df['M'].std() if df['M'].std() > 0 else 1e-9
    
    # 2. Tính Z-score cho từng Epoch
    df['Hz'] = (df['Heff'] - mu_H) / std_H
    df['Mz'] = (df['M'] - mu_M) / std_M
    
    # 3. Tính Psi Composite (Psi')
    # Công thức: Psi' = 0.5 * Hz + 0.5 * Mz
    df['Psi'] = 0.5 * df['Hz'] + 0.5 * df['Mz']
    
    return df

def plot_final_report(df):
    """
    Vẽ đồ thị 3 trục với cấu hình ÉP LỀ THỦ CÔNG (Manual Margins).
    Loại bỏ hoàn toàn khoảng trắng thừa giữa text và đồ thị.
    """
    # 1. Tắt layout tự động, chỉnh kích thước vừa phải (Chiều cao giảm xuống 5)
    plt.rcParams.update({'font.size': 11})
    fig, ax1 = plt.subplots(figsize=(12, 5)) # Không dùng layout='constrained' nữa

    # --- TRỤC 1 (TRÁI): ACCURACY ---
    color_acc = '#1f77b4' 
    ax1.set_xlabel('Epochs', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Validation Accuracy (%)', color=color_acc, fontsize=12, fontweight='bold')
    line1, = ax1.plot(df['epoch'], df['val_acc'], color=color_acc, linewidth=2.5, 
                      label='Accuracy', marker='o', markersize=3, alpha=0.9)
    ax1.tick_params(axis='y', labelcolor=color_acc)
    ax1.grid(True, linestyle='--', alpha=0.3)

    # --- TRỤC 2 (PHẢI - SÁT): PSI CHUẨN ---
    ax2 = ax1.twinx()
    color_psi = '#d62728' 
    ax2.set_ylabel('Consciousness Index ($\Psi$)', color=color_psi, fontsize=12, fontweight='bold')
    line2, = ax2.plot(df['epoch'], df['Psi'], color=color_psi, linewidth=2, 
                      linestyle='--', label='$\Psi$ Index (Z-scored)', marker='s', markersize=3)
    ax2.tick_params(axis='y', labelcolor=color_psi)

    # --- TRỤC 3 (PHẢI - XA): LOSS ---
    ax3 = ax1.twinx()
    color_loss = '#2ca02c' 
    
    # Đẩy trục này ra vị trí 1.08 (vừa đủ sát)
    ax3.spines["right"].set_position(("axes", 1.08)) 
    ax3.spines["right"].set_visible(True)
    
    ax3.set_ylabel('Validation Loss', color=color_loss, fontsize=11, fontweight='bold')
    line3, = ax3.plot(df['epoch'], df['val_loss'], color=color_loss, linewidth=1.5, 
                      linestyle=':', label='Val Loss', alpha=0.8)
    ax3.tick_params(axis='y', labelcolor=color_loss)

    # --- TRANG TRÍ ---
    # Đánh dấu Best Epoch
    best_idx = df['val_loss'].idxmin()
    best_epoch = df.loc[best_idx, 'epoch']
    
    plt.axvline(x=best_epoch, color='gray', linestyle='-.', alpha=0.5, linewidth=1)
    
    # Text nằm gọn bên trong biểu đồ, xoay dọc
    ylim_min, ylim_max = ax1.get_ylim()
    # Điều chỉnh y để text không bị bay ra ngoài
    text_y_pos = ylim_min + (ylim_max - ylim_min) * 0.1 
    ax1.text(best_epoch, text_y_pos, 
             f'  Optimal (Ep {int(best_epoch)})', 
             color='#444', fontsize=9, rotation=90, verticalalignment='bottom')

    # --- LEGEND ---
    lines = [line1, line2, line3]
    labels = [l.get_label() for l in lines]
    
    # Đưa Legend vào BÊN TRONG biểu đồ (Góc dưới phải hoặc giữa phải)
    # Để không tốn diện tích bên dưới
    ax1.legend(lines, labels, loc='lower right', 
               bbox_to_anchor=(0.95, 0.05), # Cách lề phải một chút
               fancybox=True, framealpha=0.9, shadow=True, fontsize=10)

    plt.title('Dynamics of Consciousness ($\Psi$) vs Generalization Performance', 
              fontsize=14, pad=10)
    plt.subplots_adjust(top=0.92, bottom=0.10, left=0.08, right=0.88)

    plt.show()

<>:48: SyntaxWarning: invalid escape sequence '\P'
<>:50: SyntaxWarning: invalid escape sequence '\P'
<>:91: SyntaxWarning: invalid escape sequence '\P'
<>:48: SyntaxWarning: invalid escape sequence '\P'
<>:50: SyntaxWarning: invalid escape sequence '\P'
<>:91: SyntaxWarning: invalid escape sequence '\P'
/tmp/ipykernel_163/605653552.py:48: SyntaxWarning: invalid escape sequence '\P'
  ax2.set_ylabel('Consciousness Index ($\Psi$)', color=color_psi, fontsize=12, fontweight='bold')
/tmp/ipykernel_163/605653552.py:50: SyntaxWarning: invalid escape sequence '\P'
  linestyle='--', label='$\Psi$ Index (Z-scored)', marker='s', markersize=3)
/tmp/ipykernel_163/605653552.py:91: SyntaxWarning: invalid escape sequence '\P'
  plt.title('Dynamics of Consciousness ($\Psi$) vs Generalization Performance',


In [6]:
# ---------------------------------------------------------------------
# UTILS: EARLY STOPPING (Thêm class này vào để code chạy được)
# ---------------------------------------------------------------------
class EarlyStopping:
    def __init__(self, patience=7, delta=0, path='best_model.pt'):
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'   -> EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

class FastCIFAR10(torch.utils.data.Dataset):
    def __init__(self, train=True, transform=None, device='cuda'):
        # Đường dẫn gốc trỏ thẳng vào sâu nhất nơi chứa các file batch thô
        base_path = '/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py'
        
        all_data = []
        all_targets = []
        
        if train:
            # Tập Train của CIFAR-10 được chia nhỏ làm 5 file batch độc lập
            for i in range(1, 6):
                batch_file = os.path.join(base_path, f'data_batch_{i}')
                with open(batch_file, 'rb') as f:
                    # Đọc file nhị phân bằng pickle mã hóa latin1 theo chuẩn CIFAR
                    d = pickle.load(f, encoding='latin1')
                    all_data.append(d['data'])
                    all_targets.extend(d['labels'])
            
            # Ghép 5 batch lại thành 1 ma trận duy nhất: (50000, 3072)
            raw_data = np.concatenate(all_data, axis=0)
        else:
            # Tập Validation chỉ có duy nhất 1 file test_batch
            batch_file = os.path.join(base_path, 'test_batch')
            with open(batch_file, 'rb') as f:
                d = pickle.load(f, encoding='latin1')
                raw_data = d['data']
                all_targets = d['labels']
        
        # Cấu trúc gốc của mảng d['data'] là phẳng (Flat): (N, 3072)
        # 3072 pixel gồm: 1024 pixel Đỏ, 1024 pixel Xanh Lá, 1024 pixel Xanh Dương
        # Thực hiện reshape về cấu trúc chuẩn ảnh: (N, 3, 32, 32)
        N = raw_data.shape[0]
        raw_data = raw_data.reshape(N, 3, 32, 32)
        
        # Đẩy thẳng toàn bộ mảng dữ liệu thô dạng Float lên bộ nhớ VRAM của GPU
        self.data = torch.tensor(raw_data, dtype=torch.float32, device=device)
        self.targets = torch.tensor(all_targets, dtype=torch.long, device=device)
        
        # Khởi tạo các hằng số chuẩn hóa toán học
        self.mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1).to(device)
        self.std = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1).to(device)
        self.train = train

    def __getitem__(self, index):
        # Đưa dải pixel từ [0, 255] về [0.0, 1.0] ngay trên GPU
        img = self.data[index] / 255.0
        target = self.targets[index]
        
        # Thực hiện phép toán chuẩn hóa ma trận (Broadcasting)
        img = (img - self.mean) / self.std
        
        # Thực hiện kỹ thuật Augmentation lật ảnh ngẫu nhiên để chống Overfitting
        if self.train:
            if torch.rand(1) > 0.5:
                img = torch.flip(img, dims=[2]) # Lật ảnh theo chiều ngang
                
        return img, target

    def __len__(self):
        return len(self.data)

In [7]:
# Hàm thiết lập Seed nghiêm ngặt trước mỗi lượt chạy
def set_deterministic_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Ép PyTorch tính toán chính xác tuyệt đối (chấp nhận giảm một chút tốc độ)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [11]:
# ============================================================================
# BLOCK 7: MULTI-SEED EXECUTION LOGIC (DENSENET-121)
# ============================================================================
if __name__ == "__main__":
    # 1. CẤU HÌNH HỆ THỐNG
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    VAL_BATCH_SIZE = 4096 
    TRAIN_BATCH_SIZE = 512  
    EPOCHS = 100          
    PATIENCE = 10         
    
    # Định nghĩa 5 hạt giống tiêu chuẩn học thuật cho bài báo
    SEEDS = [42, 123, 1000, 1337, 2026]
    
    all_experiments_summary = {}
    
    # SỬA ĐỒNG BỘ: Đảm bảo tất cả các key viết thường chính xác với lệnh append
    aggregated_metrics = {
        "best_accuracy": [], 
        "mean_Heff": [], 
        "std_Heff": [],
        "sigma_Psi": [], 
        "r_Hz_Mz": [], 
        "r_psi_acc": []  # Khớp hoàn toàn với aggregated_metrics["r_psi_acc"].append(...)
    }

    print(f"🚀 Bắt đầu chiến dịch huấn luyện hệ thống trên {len(SEEDS)} hạt giống (Seeds)...")

    # VÒNG LẶP ĐA HẠT GIỐNG (MULTI-SEED LOOP)
    for run_idx, current_seed in enumerate(SEEDS):
        print("\n" + "="*70)
        print(f"🔥 LƯỢT CHẠY {run_idx + 1}/{len(SEEDS)} | SEED KHỞI TẠO: {current_seed}")
        print("="*70)
        
        # Thiết lập hạt giống độc lập cho lượt chạy này
        set_deterministic_seeds(current_seed)

        # 2. DATA LOAD (Đọc trực tiếp từ VRAM qua FastCIFAR10)
        trainset_fast = FastCIFAR10(train=True, device=DEVICE)
        valset_fast = FastCIFAR10(train=False, device=DEVICE)

        trainloader = torch.utils.data.DataLoader(trainset_fast, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=0)
        valloader = torch.utils.data.DataLoader(valset_fast, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0)

        # 3. MODEL & OPTIMIZER CHUẨN HÓA CHO SEED MỚI
        model = torchvision.models.densenet121(num_classes=10).to(DEVICE)
        
        LR = 0.1
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5) 
        
        # Đóng gói checkpoint riêng cho từng seed để tránh đè file
        early_stopping = EarlyStopping(patience=PATIENCE, path=f'best_model_seed_{current_seed}.pt')
        activations = setup_densenet_hooks(model)    

        # 4. TRAINING LOOP NỘI BỘ
        history = []
        for epoch in range(EPOCHS):
            model.train()
            train_loss, correct, total = 0, 0, 0
            for inputs, targets in trainloader:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
            
            avg_train_loss = train_loss / len(trainloader)
            train_acc = 100. * correct / total

            model.eval()
            val_loss, val_correct, val_total = 0, 0, 0
            with torch.no_grad():
                for inputs, targets in valloader:
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
                    val_loss += loss.item()
                    _, predicted = outputs.max(1)
                    val_total += targets.size(0)
                    val_correct += predicted.eq(targets).sum().item()
            
            avg_val_loss = val_loss / len(valloader)
            val_acc = 100. * val_correct / val_total
            scheduler.step(avg_val_loss)

            # TÍNH PSI ĐA TẦNG (ALGORITHM 1)
            data_iter = iter(valloader)
            inputs_psi, targets_psi = next(data_iter)
            with torch.no_grad():
                _ = model(inputs_psi) 
            
            multiscale_matrix = gather_multiscale_activations(activations, samples_per_layer=32)
            cnn_signals = prepare_cnn_activations(multiscale_matrix, targets_psi, smooth_window=15)
            metrics = compute_raw_metrics(cnn_signals)

            log = {
                "epoch": epoch + 1, "train_loss": avg_train_loss, "val_loss": avg_val_loss,
                "val_acc": val_acc, "Heff": metrics["Heff"], "M": metrics["M"], "H_raw": metrics["H_raw"]
            }
            history.append(log)

            if (epoch + 1) % 10 == 0:
                print(f"   Ep {epoch+1:03d} | Loss:{avg_val_loss:.3f} | Acc:{val_acc:.2f}% | Heff:{metrics['Heff']:.2f} | M:{metrics['M']:.3f}")

            early_stopping(avg_val_loss, model)
            if early_stopping.early_stop:
                print(f"   🛑 Early stopping tại epoch {epoch+1}")
                break

        # 5. XỬ LÝ VÀ PHÂN TÍCH KẾT QUẢ RIÊNG CỦA SEED HIỆN TẠI
        df_results = summarize_experiment_results(history)
        df_valid = df_results.dropna(subset=['Heff', 'M', 'Psi', 'Hz', 'Mz']).copy()
        
        if len(df_valid) > 1:
            m_heff, s_heff = df_valid['Heff'].mean(), df_valid['Heff'].std()
            sig_psi = df_valid['Psi'].std()
            r_hz_mz = df_valid['Hz'].corr(df_valid['Mz'])
            r_psi_acc = df_valid['Psi'].corr(df_valid['val_acc'])
            b_acc = df_results['val_acc'].max()
            
            # Thu thập vào mảng tổng để tính toán liên-seed cuối cùng
            aggregated_metrics["best_accuracy"].append(b_acc)
            aggregated_metrics["mean_Heff"].append(m_heff)
            aggregated_metrics["std_Heff"].append(s_heff)
            aggregated_metrics["sigma_Psi"].append(sig_psi)
            aggregated_metrics["r_Hz_Mz"].append(r_hz_mz)
            aggregated_metrics["r_psi_acc"].append(r_psi_acc)
            
            # Lưu trữ cấu trúc record cho seed hiện tại
            seed_string = f"{b_acc:.1f}% | {m_heff:.3f} ± {s_heff:.3f} | {sig_psi:.4f} | {r_hz_mz:.3f} | {r_psi_acc:.3f}"
            all_experiments_summary[f"seed_{current_seed}"] = {
                "best_accuracy": float(b_acc), "mean_Heff": float(m_heff), "std_Heff": float(s_heff),
                "sigma_Psi": float(sig_psi), "r_Hz_Mz": float(r_hz_mz), "r_Psi_acc": float(r_psi_acc),
                "paper_ready_string": seed_string
            }
            
            # Xuất log CSV riêng cho seed này
            df_results.to_csv(f"results_densenet121_cifar10_seed_{current_seed}.csv", index=False)
            print(f"✅ Đã lưu kết quả chi tiết của Seed {current_seed}")
        
        # GIẢI PHÓNG BỘ NHỚ VRAM TUYỆT ĐỐI TRƯỚC KHI SANG SEED MỚI
        del model, optimizer, trainloader, valloader
        gc.collect()
        if torch.cuda.is_available(): 
            torch.cuda.empty_cache()

    # ============================================================================
    # 6. TÍNH TOÁN SAI SỐ TOÀN CỤC LIÊN-SEED (MEAN ± STD OF SEEDS)
    # ============================================================================
    print("\n" + "🏁" * 15 + " KẾT LUẬN TOÀN DIỆN DIỄN ĐÀN 5 SEEDS " + "🏁" * 15)
    
    final_report = {
        "metadata": {"architecture": "DenseNet-121", "dataset": "CIFAR-10", "seeds_tested": SEEDS}
    }
    
    # Tính toán Mean và Std của chính các phân phối giữa các lượt chạy Seed khác nhau
    summary_outputs = {}
    for k, values in aggregated_metrics.items():
        if values:
            arr = np.array(values)
            summary_outputs[k] = {"mean": float(arr.mean()), "std": float(arr.std())}
            
    final_report["global_statistics"] = summary_outputs
    final_report["individual_runs"] = all_experiments_summary

    # Tạo chuỗi định dạng tối cao kết hợp đầy đủ sai số giữa các hạt giống cho bài báo
    super_paper_line = (
        f"DenseNet-121 CIFAR-10 | "
        f"Best Acc: {summary_outputs['best_accuracy']['mean']:.1f}±{summary_outputs['best_accuracy']['std']:.1f}% | "
        f"Heff: {summary_outputs['mean_Heff']['mean']:.3f}±{summary_outputs['mean_Heff']['std']:.3f} | "
        f"σ_Ψ: {summary_outputs['sigma_Psi']['mean']:.4f}±{summary_outputs['sigma_Psi']['std']:.4f} | "
        f"r(Hz,Mz): {summary_outputs['r_Hz_Mz']['mean']:.3f}±{summary_outputs['r_Hz_Mz']['std']:.3f} | "
        f"r(Ψ,acc): {summary_outputs['r_psi_acc']['mean']:.3f}±{summary_outputs['r_psi_acc']['std']:.3f}"
    )
    final_report["super_paper_ready_string"] = super_paper_line

    print("\n📊 DÒNG TỔNG HỢP SIÊU ĐẠT CHUẨN CHO KHUNG BIỆN MINH PHẢN BIỆN 1:")
    print("-" * 100)
    print(super_paper_line)
    print("-" * 100)

    # Xuất file JSON tối cao chứa cấu trúc của cả 5 lần chạy
    with open("multi_seed_summary_densenet121_cifar10.json", "w", encoding="utf-8") as f:
        json.dump(final_report, f, ensure_ascii=False, indent=4)
    print("\n💾 Đã đóng gói toàn diện 5 hạt giống vào file: multi_seed_summary_densenet121_cifar10.json")

🚀 Bắt đầu chiến dịch huấn luyện hệ thống trên 5 hạt giống (Seeds)...

🔥 LƯỢT CHẠY 1/5 | SEED KHỞI TẠO: 42
Hooking DenseNet layers: ['denseblock1', 'denseblock2', 'denseblock3', 'denseblock4']
   -> EarlyStopping counter: 1 out of 10
   Ep 010 | Loss:0.724 | Acc:75.21% | Heff:0.92 | M:0.087
   -> EarlyStopping counter: 1 out of 10
   -> EarlyStopping counter: 2 out of 10
   -> EarlyStopping counter: 3 out of 10
   -> EarlyStopping counter: 4 out of 10
   -> EarlyStopping counter: 5 out of 10
   -> EarlyStopping counter: 6 out of 10
   -> EarlyStopping counter: 7 out of 10
   -> EarlyStopping counter: 8 out of 10
   -> EarlyStopping counter: 9 out of 10
   Ep 020 | Loss:0.932 | Acc:79.81% | Heff:0.93 | M:0.054
   -> EarlyStopping counter: 10 out of 10
   🛑 Early stopping tại epoch 20
✅ Đã lưu kết quả chi tiết của Seed 42

🔥 LƯỢT CHẠY 2/5 | SEED KHỞI TẠO: 123
Hooking DenseNet layers: ['denseblock1', 'denseblock2', 'denseblock3', 'denseblock4']
   -> EarlyStopping counter: 1 out of 10
   -

/tmp/ipykernel_163/2690917344.py:56: RuntimeWarning: divide by zero encountered in log
  coeffs = np.polyfit(np.log(s_use), np.log(F), 1)


   -> EarlyStopping counter: 1 out of 10
   Ep 010 | Loss:0.785 | Acc:72.55% | Heff:0.71 | M:0.072


/tmp/ipykernel_163/2690917344.py:56: RuntimeWarning: divide by zero encountered in log
  coeffs = np.polyfit(np.log(s_use), np.log(F), 1)


   -> EarlyStopping counter: 1 out of 10
   -> EarlyStopping counter: 2 out of 10


/tmp/ipykernel_163/2690917344.py:56: RuntimeWarning: divide by zero encountered in log
  coeffs = np.polyfit(np.log(s_use), np.log(F), 1)


   -> EarlyStopping counter: 3 out of 10
   -> EarlyStopping counter: 4 out of 10


/tmp/ipykernel_163/2690917344.py:56: RuntimeWarning: divide by zero encountered in log
  coeffs = np.polyfit(np.log(s_use), np.log(F), 1)


   -> EarlyStopping counter: 5 out of 10
   -> EarlyStopping counter: 6 out of 10
   -> EarlyStopping counter: 7 out of 10
   -> EarlyStopping counter: 8 out of 10
   Ep 020 | Loss:0.955 | Acc:75.24% | Heff:0.71 | M:0.088
   -> EarlyStopping counter: 9 out of 10
   -> EarlyStopping counter: 10 out of 10
   🛑 Early stopping tại epoch 21
✅ Đã lưu kết quả chi tiết của Seed 1000

🔥 LƯỢT CHẠY 4/5 | SEED KHỞI TẠO: 1337
Hooking DenseNet layers: ['denseblock1', 'denseblock2', 'denseblock3', 'denseblock4']
   -> EarlyStopping counter: 1 out of 10
   -> EarlyStopping counter: 1 out of 10
   Ep 010 | Loss:0.759 | Acc:73.70% | Heff:0.91 | M:0.083
   -> EarlyStopping counter: 1 out of 10
   -> EarlyStopping counter: 1 out of 10
   -> EarlyStopping counter: 2 out of 10
   -> EarlyStopping counter: 3 out of 10
   -> EarlyStopping counter: 4 out of 10
   -> EarlyStopping counter: 5 out of 10
   -> EarlyStopping counter: 6 out of 10
   -> EarlyStopping counter: 7 out of 10
   Ep 020 | Loss:0.934 | Acc:

NameError: name 'json' is not defined

In [13]:
import json
# ============================================================================
# 6. TÍNH TOÁN SAI SỐ TOÀN CỤC LIÊN-SEED (MEAN ± STD OF SEEDS)
# ============================================================================
print("\n" + "🏁" * 15 + " KẾT LUẬN TOÀN DIỆN DIỄN ĐÀN 5 SEEDS " + "🏁" * 15)

final_report = {
    "metadata": {"architecture": "DenseNet-121", "dataset": "CIFAR-10", "seeds_tested": SEEDS}
}

# Tính toán Mean và Std của chính các phân phối giữa các lượt chạy Seed khác nhau
summary_outputs = {}
for k, values in aggregated_metrics.items():
    if values:
        arr = np.array(values)
        summary_outputs[k] = {"mean": float(arr.mean()), "std": float(arr.std())}
        
final_report["global_statistics"] = summary_outputs
final_report["individual_runs"] = all_experiments_summary

# Tạo chuỗi định dạng tối cao kết hợp đầy đủ sai số giữa các hạt giống cho bài báo
super_paper_line = (
    f"DenseNet-121 CIFAR-10 | "
    f"Best Acc: {summary_outputs['best_accuracy']['mean']:.1f}±{summary_outputs['best_accuracy']['std']:.1f}% | "
    f"Heff: {summary_outputs['mean_Heff']['mean']:.3f}±{summary_outputs['mean_Heff']['std']:.3f} | "
    f"σ_Ψ: {summary_outputs['sigma_Psi']['mean']:.4f}±{summary_outputs['sigma_Psi']['std']:.4f} | "
    f"r(Hz,Mz): {summary_outputs['r_Hz_Mz']['mean']:.3f}±{summary_outputs['r_Hz_Mz']['std']:.3f} | "
    f"r(Ψ,acc): {summary_outputs['r_psi_acc']['mean']:.3f}±{summary_outputs['r_psi_acc']['std']:.3f}"
)
final_report["super_paper_ready_string"] = super_paper_line

print("\n📊 DÒNG TỔNG HỢP SIÊU ĐẠT CHUẨN CHO KHUNG BIỆN MINH PHẢN BIỆN 1:")
print("-" * 100)
print(super_paper_line)
print("-" * 100)

# Xuất file JSON tối cao chứa cấu trúc của cả 5 lần chạy
with open("multi_seed_summary_densenet121_cifar10.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, ensure_ascii=False, indent=4)
print("\n💾 Đã đóng gói toàn diện 5 hạt giống vào file: multi_seed_summary_densenet121_cifar10.json")


🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁 KẾT LUẬN TOÀN DIỆN DIỄN ĐÀN 5 SEEDS 🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁🏁

📊 DÒNG TỔNG HỢP SIÊU ĐẠT CHUẨN CHO KHUNG BIỆN MINH PHẢN BIỆN 1:
----------------------------------------------------------------------------------------------------
DenseNet-121 CIFAR-10 | Best Acc: 77.9±1.2% | Heff: 0.832±0.067 | σ_Ψ: 0.6243±0.0636 | r(Hz,Mz): -0.191±0.156 | r(Ψ,acc): 0.414±0.180
----------------------------------------------------------------------------------------------------

💾 Đã đóng gói toàn diện 5 hạt giống vào file: multi_seed_summary_densenet121_cifar10.json
